In [8]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

In [9]:
# Connect to existing collection
chroma_client = chromadb.PersistentClient(path="./chroma")
emission_db = chroma_client.get_collection(name="bitcoin_mining_emissions")

In [10]:
@function_tool
def emission_lookup_tool(query: str, max_results: int = 5) -> str:
    """
    Tool function for the internal RAG database to look up emission data collection 
    activities for Bitcoin mining operations (Scope 1, 2, and 3).
    
    Use this tool for:
    - Standard emission tracking activities
    - Data collection requirements
    - Units of measurement
    - Typical percentage impacts
    
    Use Exa Search instead for:
    - Current emission factors
    - Latest regulations
    - Real-world case studies
    - Technology updates

    Args:
        query: The emission activity or category to look up (e.g., "electricity", 
               "ASIC miners", "cooling systems", "Scope 2").
        max_results: The maximum number of results to return (default: 5).

    Returns:
        A string containing the emission tracking information from the database.
    """
    results = emission_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No emission tracking information found for: {query}"

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope = metadata["emission_scope"].upper()
        category = metadata["activity_category"].replace("_", " ").title()
        data_to_collect = metadata["data_to_collect"]
        unit = metadata["unit_of_measure"]
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()

        formatted_results.append(
            f"""
{i+1}. {emission_source}
   - Scope: {scope}
   - Category: {category}
   - Data to collect: {data_to_collect}
   - Unit: {unit}
   - Typical % of emissions: {percentage}%
   - Priority: {priority}
            """.strip()
        )

    return "Emission Tracking Activities (from internal database):\n\n" + "\n\n".join(formatted_results)


In [11]:
@function_tool
def emission_filter_tool(
    scope: str = None, 
    priority_level: str = None, 
    min_percentage: float = None,
    max_results: int = 10
) -> str:
    """
    Tool function to filter emission activities by specific criteria from the internal database.
    
    Use this for precise filtering when you need activities matching specific criteria.

    Args:
        scope: Filter by emission scope - options: "scope1", "scope2", "scope3"
        priority_level: Filter by priority - options: "critical", "high", "medium", "low-medium", "low"
        min_percentage: Filter by minimum percentage impact (e.g., 1.0 for activities >1%)
        max_results: The maximum number of results to return (default: 10)

    Returns:
        A string containing filtered emission tracking information.
    """
    where_clause = {}
    
    if scope:
        where_clause["emission_scope"] = scope.lower()
    
    if priority_level:
        where_clause["priority_level"] = priority_level.lower()
    
    if min_percentage is not None:
        where_clause["percentage_min"] = {"$gte": min_percentage}
    
    query_parts = []
    if scope:
        query_parts.append(f"{scope} emissions")
    if priority_level:
        query_parts.append(f"{priority_level} priority")
    if min_percentage:
        query_parts.append(f"high impact activities")
    
    query_text = " ".join(query_parts) if query_parts else "emission activities"
    
    results = emission_db.query(
        query_texts=[query_text], 
        n_results=max_results,
        where=where_clause if where_clause else None
    )

    if not results["documents"][0]:
        return f"No emission activities found matching the criteria."

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope_label = metadata["emission_scope"].upper()
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()
        unit = metadata["unit_of_measure"]

        formatted_results.append(
            f"{i+1}. {emission_source} ({scope_label}) - {percentage}% impact - Priority: {priority} - Unit: {unit}"
        )

    return "Filtered Emission Activities (from internal database):\n\n" + "\n".join(formatted_results)

In [5]:
# Setup Exa Search MCP Server
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 30,
    },
    client_session_timeout_seconds=30,
    cache_tools_list=True,
    max_retry_attempts=1,
)

await exa_search_mcp.connect()

# Bitcoin Mining Emissions Agent with Search Support
emission_agent_with_search = Agent(
    name="Bitcoin Mining Emissions Assistant with Web Search",
    instructions="""
    You are a specialized assistant helping with carbon accounting and emissions tracking 
    for Bitcoin mining operations. You provide guidance on Scope 1, Scope 2, and Scope 3 
    emission data collection activities according to the GHG Protocol.

    **Your workflow:**
    0) First, use emission_lookup_tool or emission_filter_tool to check the internal database 
       for emission tracking activities and guidance.
    1) If the query requires current industry practices, regulatory updates, emission factors, 
       or real-world examples that may not be in the database, use Exa Search to find:
       - Latest emission factors for electricity grids
       - Current best practices in Bitcoin mining sustainability
       - Regulatory requirements and compliance frameworks
       - Case studies of mining operations
       - Renewable energy integration examples
       - Technology updates (cooling systems, ASIC efficiency)
    2) After web search, cross-reference findings with emission_lookup_tool to ensure 
       consistency with GHG Protocol standards.
    3) Combine database knowledge with current web information for comprehensive answers.

    **Key facts to remember:**
    - Scope 1 = Direct emissions (on-site combustion, fugitive emissions)
    - Scope 2 = Indirect emissions from purchased electricity
    - Scope 3 = Other indirect emissions (supply chain, capital goods, etc.)
    - ASIC miner electricity typically represents 95-98% of total emissions (CRITICAL priority)
    - Cooling systems represent 1-3% of emissions
    - PUE (Power Usage Effectiveness) target: <1.2 for efficient operations

    **When to use each tool:**
    - emission_lookup_tool: For standard emission tracking activities and data collection requirements
    - emission_filter_tool: For filtering activities by scope, priority, or impact level
    - Exa Search: For current emission factors, regulations, case studies, and industry updates

    **Output guidelines:**
    - Always mention priority level and typical percentage impact for activities
    - Cite sources when using web search results
    - Include specific units of measurement
    - Provide actionable, technical guidance
    - Keep answers concise but comprehensive
    - Don't use emission tools more than 10 times per query
    """,
    tools=[emission_lookup_tool, emission_filter_tool],
    mcp_servers=[exa_search_mcp],
)

In [6]:
with trace("Regulatory Compliance"):
    result = await Runner.run(
        emission_agent_with_search,
        """What are the latest carbon reporting regulations for cryptocurrency mining 
        in the EU, and what emission data do I need to collect to comply?"""
    )
    print(result.final_output)

Here are the latest EU requirements and the emission data you should collect for crypto mining compliance. I’ve split regulatory obligations and data collection needs, with concise action items.

1) Regulatory landscape (EU)

- MiCA sustainability disclosures (EU Regulation 2023/1114)
  - What it covers: For crypto-asset issuers and service providers, MiCA imposes transparency on environmental/climate impacts and other sustainability disclosures. First obligations applicable from 30 June 2024 (with ongoing requirements as MiCA is phased in).
  - Relevance to mining operators: If your entity falls under MiCA (e.g., you issue/trade crypto-assets or operate crypto-asset services under MiCA, depending on your business model), you must disclose climate/environmental impact information in a consistent, comparable format.
  - Practical impact: Prepare and publish climate-related disclosures for transparent reporting to authorities and users.

  Sources:
  - Commission Delegated Regulation (EU

In [7]:
with trace("Current Grid Emission Factors"):
    result = await Runner.run(
        emission_agent_with_search,
        """What are the current grid emission factors for Indonesia, and how should 
        I apply them to track Scope 2 emissions from my Bitcoin mining operation?"""
    )
    print(result.final_output)

Here’s how to handle Scope 2 for Indonesia and what to use for grid EF.

What to use
- Use the latest Indonesia grid emission factor (EF): CO2e per kWh from the national grid. This is typically reported as gCO2e/kWh (convert to kgCO2e/kWh by dividing by 1000).
- Good live references (as of now): Our World in Data carbon intensity of electricity (gCO2e/kWh) and other recent national/regional grid reports. If you need a country-specific figure for a precise period (e.g., 2023 vs 2024), pull the most recent value from a credible source (OWID, IEA, national energy plan reports, or utility/regulated grid data).

How to apply (Scope 2 for a Bitcoin mining operation)
- Step 1: Determine annual electricity purchased for mining operations (kWh).
  - Include ASIC miner electricity, cooling, and any supporting facility electricity that is directly used for mining.
  - Exclude on-site renewable production you self-consume unless you are calculating market-based scope 2 (see below).
- Step 2: Obtai

In [ ]:
with trace("Industry Best Practices"):
    result = await Runner.run(
        emission_agent_with_search,
        """Show me examples of Bitcoin mining operations that have successfully 
        implemented renewable energy integration. What emission tracking activities 
        did they prioritize?"""
    )
    print(result.final_output)

In [13]:
with trace("Complete Setup"):
    result = await Runner.run(
        emission_agent_with_search,
        """I'm setting up a new 10 MW Bitcoin mining facility in Indonesia. 
        What are the critical emission tracking activities I need to implement, 
        and what are current industry best practices for similar operations?"""
    )
    print(result.final_output)

Here’s a concise, action-oriented plan tailored to a 10 MW Bitcoin mining facility in Indonesia, based on your internal emissions guidance and current industry context.

1) Critical emission tracking activities (Scope 1, 2, and 3)
- ASIC miner electricity consumption
  - Scope: Scope 2
  - Data to collect: real-time and historical electricity usage for all installed miners (kWh)
  - Typical impact: 95–98% of total emissions
  - Priority: CRITICAL
  - Why: ASICs dominate electricity-related emissions; you must capture device-level energy demand to establish baseline, monitor efficiency, and drive reductions.

- Facility-wide purchased electricity
  - Scope: Scope 2
  - Data to collect: total facility electricity usage (kWh), time-of-use, and supplier contracts/hydrocarbon content
  - Typical impact: 0.1–0.5% (relative to miner-level emissions, but essential for Scope 2 accounting)
  - Priority: HIGH
  - Why: Necessary for complete Scope 2 accounting and for any renewable energy/green el

In [14]:
with trace("Emission Factors"):
    result = await Runner.run(
        emission_agent_with_search,
        """I'm using grid electricity in Jakarta. What's the current emission factor 
        in gCO2/kWh, and what Scope 2 activities should I track according to GHG Protocol?"""
    )
    print(result.final_output)

Short answer:
- Current grid emission factor for Jakarta (gCO2/kWh): around 600–750 gCO2/kWh based on Indonesia’s power mix (industrial estimates place Indonesia’s grid intensity well above 600 gCO2/kWh in recent years; Ember reports ~623 gCO2/kWh for 2021). If you need a precise, location-specific figure for today, use a grid-factor data source (e.g., local utility residuals, or a reputable grid emissions dataset) and report both location-based and market-based values. See sources cited below. ([ember-energy.org](https://ember-energy.org/chapter/country-and-region-deep-dives-ger2023/?utm_source=openai))

- Scope 2 activities to track (per GHG Protocol):
  1) Purchased electricity (on-site mining electricity and any auxiliary electrical loads)
     - Data to collect: monthly kWh consumed, meter readings, and supplier/invoice data
     - Unit: kWh
     - Emission factor application: use location-based grid factor (gCO2/kWh) for location-based Scope 2; if you have contracts/RECs, also co

## Native Web Search

In [18]:
# Connect to existing collection
chroma_client = chromadb.PersistentClient(path="./chroma")
emission_db = chroma_client.get_collection(name="bitcoin_mining_emissions")

In [20]:
@function_tool
def emission_lookup_tool(query: str, max_results: int = 5) -> str:
    """
    Tool function for the internal RAG database to look up emission data collection 
    activities for Bitcoin mining operations (Scope 1, 2, and 3).
    
    Use this tool for:
    - Standard emission tracking activities
    - Data collection requirements
    - Units of measurement
    - Typical percentage impacts
    
    Use web search instead for:
    - Current emission factors
    - Latest regulations
    - Real-world case studies
    - Technology updates

    Args:
        query: The emission activity or category to look up (e.g., "electricity", 
               "ASIC miners", "cooling systems", "Scope 2").
        max_results: The maximum number of results to return (default: 5).

    Returns:
        A string containing the emission tracking information from the database.
    """
    results = emission_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No emission tracking information found for: {query}"

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope = metadata["emission_scope"].upper()
        category = metadata["activity_category"].replace("_", " ").title()
        data_to_collect = metadata["data_to_collect"]
        unit = metadata["unit_of_measure"]
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()

        formatted_results.append(
            f"""
{i+1}. {emission_source}
   - Scope: {scope}
   - Category: {category}
   - Data to collect: {data_to_collect}
   - Unit: {unit}
   - Typical % of emissions: {percentage}%
   - Priority: {priority}
            """.strip()
        )

    return "Emission Tracking Activities (from internal database):\n\n" + "\n\n".join(formatted_results)


In [21]:
@function_tool
def emission_filter_tool(
    scope: str = None, 
    priority_level: str = None, 
    min_percentage: float = None,
    max_results: int = 10
) -> str:
    """
    Tool function to filter emission activities by specific criteria from the internal database.
    
    Use this for precise filtering when you need activities matching specific criteria.

    Args:
        scope: Filter by emission scope - options: "scope1", "scope2", "scope3"
        priority_level: Filter by priority - options: "critical", "high", "medium", "low-medium", "low"
        min_percentage: Filter by minimum percentage impact (e.g., 1.0 for activities >1%)
        max_results: The maximum number of results to return (default: 10)

    Returns:
        A string containing filtered emission tracking information.
    """
    where_clause = {}
    
    if scope:
        where_clause["emission_scope"] = scope.lower()
    
    if priority_level:
        where_clause["priority_level"] = priority_level.lower()
    
    if min_percentage is not None:
        where_clause["percentage_min"] = {"$gte": min_percentage}
    
    query_parts = []
    if scope:
        query_parts.append(f"{scope} emissions")
    if priority_level:
        query_parts.append(f"{priority_level} priority")
    if min_percentage:
        query_parts.append(f"high impact activities")
    
    query_text = " ".join(query_parts) if query_parts else "emission activities"
    
    results = emission_db.query(
        query_texts=[query_text], 
        n_results=max_results,
        where=where_clause if where_clause else None
    )

    if not results["documents"][0]:
        return f"No emission activities found matching the criteria."

    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        
        emission_source = metadata["emission_source"].title()
        scope_label = metadata["emission_scope"].upper()
        percentage = metadata["percentage_range"]
        priority = metadata["priority_level"].upper()
        unit = metadata["unit_of_measure"]

        formatted_results.append(
            f"{i+1}. {emission_source} ({scope_label}) - {percentage}% impact - Priority: {priority} - Unit: {unit}"
        )

    return "Filtered Emission Activities (from internal database):\n\n" + "\n".join(formatted_results)


In [22]:
# Bitcoin Mining Emissions Agent with Web Search Support
emission_agent_with_search = Agent(
    name="Bitcoin Mining Emissions Assistant with Web Search",
    instructions="""
    You are a specialized assistant helping with carbon accounting and emissions tracking 
    for Bitcoin mining operations. You provide guidance on Scope 1, Scope 2, and Scope 3 
    emission data collection activities according to the GHG Protocol.

    **Your workflow:**
    0) First, use emission_lookup_tool or emission_filter_tool to check the internal database 
       for emission tracking activities and guidance.
    1) If the query requires current industry practices, regulatory updates, emission factors, 
       or real-world examples that may not be in the database, use web search to find:
       - Latest emission factors for electricity grids
       - Current best practices in Bitcoin mining sustainability
       - Regulatory requirements and compliance frameworks
       - Case studies of mining operations
       - Renewable energy integration examples
       - Technology updates (cooling systems, ASIC efficiency)
       - Current grid carbon intensity data
    2) After web search, always cross-reference findings with emission_lookup_tool to ensure 
       consistency with GHG Protocol standards.
    3) Combine database knowledge with current web information for comprehensive answers.

    **Key facts to remember:**
    - Scope 1 = Direct emissions (on-site combustion, fugitive emissions)
    - Scope 2 = Indirect emissions from purchased electricity
    - Scope 3 = Other indirect emissions (supply chain, capital goods, etc.)
    - ASIC miner electricity typically represents 95-98% of total emissions (CRITICAL priority)
    - Cooling systems represent 1-3% of emissions
    - PUE (Power Usage Effectiveness) target: <1.2 for efficient operations

    **When to use each tool:**
    - emission_lookup_tool: For standard emission tracking activities and data collection requirements
    - emission_filter_tool: For filtering activities by scope, priority, or impact level
    - web_search: For current emission factors, regulations, case studies, and industry updates

    **Output guidelines:**
    - Always mention priority level and typical percentage impact for activities
    - Cite sources when using web search results
    - Include specific units of measurement
    - Provide actionable, technical guidance
    - Keep answers concise but comprehensive
    - Don't use emission_lookup_tool or emission_filter_tool more than 8 times per query
    """,
    tools=[emission_lookup_tool, emission_filter_tool, WebSearchTool()],
)

In [23]:
# Example 1: Query requiring both database and web search
with trace("Current Grid Emission Factors"):
    result = await Runner.run(
        emission_agent_with_search,
        """What are the current grid emission factors for Indonesia (Jakarta specifically), 
        and how should I apply them to track Scope 2 emissions from my Bitcoin mining operation?"""
    )
    print(result.final_output)

Here are the current (most-recent available) grid emission factors for Indonesia (Jakarta region context) and how to apply them to Scope 2 tracking for Bitcoin mining.

What to use now (fact box)
- Primary grid EF (Indonesia, general): ~0.78 kg CO2e per kWh (0.7848 kg CO2e/kWh) based on IEA-derived grid factors used in Climatiq (region Indonesia, year 2023 data, production-mix context; published as 2021-era factor re-listed in 2024 datasets). This appears as an Indonesia-wide grid factor, suitable if you don’t have a Jakarta-specific factor. Source: Climatiq data page referencing IEA Life Cycle Upstream Emission Factors (Indonesia). 0.7848 kg CO2e/kWh. ([climatiq.io](https://www.climatiq.io/data/emission-factor/b6e5806a-cbb5-4daa-b2d8-d4ed004f4e95?utm_source=openai))
- Alternative corroboration (country-level): Climate Transparency / Climate Trade data cited in Climatiq (same ballpark around 0.78–0.79 kg CO2e/kWh for Indonesia in recent years). 0.7848 kg CO2e/kWh. ([climatiq.io](https:

In [24]:
with trace("Cooling Technology"):
    result = await Runner.run(
        emission_agent_with_search,
        """What are the latest immersion cooling technologies for ASIC miners, 
        and how do they affect refrigerant emissions (Scope 1)? What data should I track?"""
    )
    print(result.final_output)

Here’s a concise update tailored to Bitcoin mining with immersion cooling, plus what to track for Scope 1 emissions.

1) Latest immersion cooling technologies for ASIC miners (highlights)
- Two-phase immersion cooling with dielectric refrigerants
  - What it is: heat is carried away by a dielectric fluid that boils/condenses in a closed loop, enabling very high heat removal density.
  - Relevance: increasingly deployed in mining and HPC; enables higher ASIC density and better energy effectiveness than single-phase fluids. Examples and industry mentions exist for 2-phase systems in mining and data centers. ([cayenneindustrytimes.com](https://www.cayenneindustrytimes.com/article/807695071-immersion-cooling-market-valued-at-usd-400-million-in-2024-projected-to-grow-at-22-5-cagr-through-2035?utm_source=openai))
- Direct-to-chip immersion variants (single-phase immersion as alternative to full tanks)
  - What it is: coolant contacts cold plates/chips directly; simpler maintenance than full 